# EDA — ASL Alphabet Dataset
**SignBridge | Day 1**

Explores the Kaggle ASL Alphabet dataset (87,000 images, 29 classes).

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from collections import Counter
from google.cloud import storage
import io
from PIL import Image

# GCS config
BUCKET = 'signbridge-data'
TRAIN_PREFIX = 'raw/asl_alphabet/train/asl_alphabet_train/'
TEST_PREFIX  = 'raw/asl_alphabet/test/asl_alphabet_test/'
FIGURES_DIR  = '../docs/figures/'
os.makedirs(FIGURES_DIR, exist_ok=True)

client = storage.Client()
bucket = client.bucket(BUCKET)
print('GCS connected:', BUCKET)

## 1. Class Distribution

In [ ]:
# Count images per class by listing GCS prefixes
blobs = client.list_blobs(BUCKET, prefix=TRAIN_PREFIX, delimiter='/')
_ = list(blobs)  # consume iterator to populate prefixes
class_prefixes = [p.split('/')[-2] for p in blobs.prefixes]
print(f'Classes found: {len(class_prefixes)}')
print(sorted(class_prefixes))

In [ ]:
# Count files per class
class_counts = {}
for cls in sorted(class_prefixes):
    prefix = TRAIN_PREFIX + cls + '/'
    blobs_cls = client.list_blobs(BUCKET, prefix=prefix)
    class_counts[cls] = sum(1 for _ in blobs_cls)
    print(f'  {cls}: {class_counts[cls]}')

print(f'\nTotal train images: {sum(class_counts.values())}')

In [ ]:
# Plot class distribution
classes = sorted(class_counts.keys())
counts  = [class_counts[c] for c in classes]

fig, ax = plt.subplots(figsize=(14, 5))
bars = ax.bar(classes, counts, color='steelblue', edgecolor='white')
ax.set_xlabel('ASL Class')
ax.set_ylabel('Image Count')
ax.set_title('ASL Alphabet — Training Set Class Distribution')
ax.axhline(np.mean(counts), color='red', linestyle='--', label=f'Mean: {np.mean(counts):.0f}')
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR + 'asl_class_dist.png', dpi=150)
plt.show()
print('Saved: docs/figures/asl_class_dist.png')

## 2. Sample Images Per Class

In [ ]:
def load_image_from_gcs(blob_name):
    blob = bucket.blob(blob_name)
    img_bytes = blob.download_as_bytes()
    return Image.open(io.BytesIO(img_bytes)).convert('RGB')

# Show 1 sample per class (first 29 classes)
fig, axes = plt.subplots(3, 10, figsize=(20, 7))
axes = axes.flatten()

for i, cls in enumerate(classes[:29]):
    prefix = TRAIN_PREFIX + cls + '/'
    blob_list = list(client.list_blobs(BUCKET, prefix=prefix, max_results=1))
    if blob_list:
        img = load_image_from_gcs(blob_list[0].name)
        axes[i].imshow(img)
        axes[i].set_title(cls, fontsize=9)
    axes[i].axis('off')

# hide unused subplots
for j in range(len(classes), len(axes)):
    axes[j].axis('off')

plt.suptitle('ASL Alphabet — 1 Sample Per Class', fontsize=14)
plt.tight_layout()
plt.savefig(FIGURES_DIR + 'asl_samples.png', dpi=150)
plt.show()
print('Saved: docs/figures/asl_samples.png')

## 3. Image Size & Quality Check

In [ ]:
# Sample 5 images per class and check dimensions
sizes = []
for cls in classes:
    prefix = TRAIN_PREFIX + cls + '/'
    blob_list = list(client.list_blobs(BUCKET, prefix=prefix, max_results=5))
    for blob in blob_list:
        img = load_image_from_gcs(blob.name)
        sizes.append(img.size)  # (width, height)

unique_sizes = Counter(sizes)
print('Unique image sizes (W x H):')
for size, count in unique_sizes.most_common(10):
    print(f'  {size}: {count} images')

## 4. EDA Findings Summary

In [ ]:
print('=== ASL EDA FINDINGS ===')
print(f'Total classes:       {len(class_counts)}')
print(f'Total train images:  {sum(class_counts.values())}')
print(f'Min class count:     {min(counts)} ({classes[np.argmin(counts)]})')
print(f'Max class count:     {max(counts)} ({classes[np.argmax(counts)]})')
print(f'Mean per class:      {np.mean(counts):.0f}')
print(f'Std dev:             {np.std(counts):.0f}')
print(f'Most common size:    {unique_sizes.most_common(1)[0][0]}')
print('')
print('Notes for report_log.md:')
print('  - Check if J and Z are present (they require motion — may be excluded)')
print('  - Check background diversity (mostly white/clean = needs augmentation)')
print('  - Class imbalance: if std/mean > 0.1, apply class weights')